In [ ]:
import sys
from pathlib import Path

# notebook/jupyter  →  proyecto/
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from scr.python.parquet_store import ParquetStore
from scr.python.parquets_derivados import (
    # Función de utilidad
    meses_disponibles,
    
    # Función principal del pipeline
    process_month,
    
    # Dominios derivados
    euros_pais,
    euros_sectores,
    euros_sectores_pais,
    euros_taric,
    euros_taric_pais,
    kg_pais,
    kg_taric,
    kg_taric_pais,
)

%load_ext autoreload
%autoreload 2

In [ ]:
# Últimos años y último mes disponible
ultimo_anio_prov = 2025
ultimo_mes_disponible = 10 
ultimo_anio_def = 2023

# Rangos de años calculados automáticamente
years_def = range(1995, ultimo_anio_def + 1)         
years_prov = range(2024, ultimo_anio_prov + 1)       

# Ámbitos
ambitos = ["espana", "madrid"]

# -------------------------------
# PIPELINE DE DOMINIOS DERIVADOS
# -------------------------------
PIPELINE = [
    # -------- TARIC
    ("taric", "euros_taric",       euros_taric),
    ("taric", "euros_taric_pais",  euros_taric_pais),
    ("taric", "euros_pais",        euros_pais),
    ("taric", "kg_taric",          kg_taric),
    ("taric", "kg_taric_pais",     kg_taric_pais),
    ("taric", "kg_pais",           kg_pais),

    # -------- SECTORES
    ("sectores", "euros_sectores",       euros_sectores),
    ("sectores", "euros_sectores_pais",  euros_sectores_pais),
]


# -------------------------------
# INICIALIZAR STORE
# -------------------------------
store = ParquetStore(
    "/home/pivan/comercio_exterior_bienes/data/rawparquet"
)

# -------------------------------
# EJECUCIÓN COMPLETA
# -------------------------------
for ambito in ambitos:
    for dominio_in, dominio_out, transform in PIPELINE:

        # -------- estado provisional (0)
        for anio in years_prov:
            for mes in meses_disponibles(anio, ultimo_anio_prov, ultimo_mes_disponible):
                print(f"🔹 Procesando: estado=0 | ambito={ambito} | dominio={dominio_out} | año={anio} | mes={mes}")
                process_month(
                    store,
                    ambito=ambito,
                    dominio_in=dominio_in,
                    dominio_out=dominio_out,
                    estado=0,
                    anio=anio,
                    mes=mes,
                    transform=transform,
                )

        # -------- estado definitivo (1)
        for anio in years_def:
            for mes in meses_disponibles(anio, ultimo_anio_prov, ultimo_mes_disponible):
                print(f"🔹 Procesando: estado=1 | ambito={ambito} | dominio={dominio_out} | año={anio} | mes={mes}")
                process_month(
                    store,
                    ambito=ambito,
                    dominio_in=dominio_in,
                    dominio_out=dominio_out,
                    estado=1,
                    anio=anio,
                    mes=mes,
                    transform=transform,
                )

In [3]:
# -------------------------------
# LECTURA PARQUET
# -------------------------------

# Parámetros del archivo a leer
ambito = "espana"
dominio = "euros_taric_pais"   
estado = 0                 
anio = 2025
mes = 10                  
filename = "data_0.parquet"

# Inicializar store
store = ParquetStore("/home/pivan/comercio_exterior_bienes/data/rawparquet")

# Leer el parquet derivado
try:
    df = store.read(
        ambito=ambito,
        dominio=dominio,
        estado=estado,
        anio=anio,
        mes=mes,
        filename=filename,
    )

    # Mostrar información básica
    print(f"Archivo leído: {store.build_path(ambito, dominio, estado, anio, mes, filename)}")
    print("\nColumnas y tipos:")
    for col, tipo in df.schema.items():
        print(f"{col}: {tipo}")
    
    print("\nPrimeras filas:")
    print(df.head(10))

except FileNotFoundError:
    print(f"⚠️ El archivo no existe: {store.build_path(ambito, dominio, estado, anio, mes, filename)}")

df


Archivo leído: /home/pivan/comercio_exterior_bienes/data/rawparquet/espana/euros_taric_pais/estado=0/anio=2025/mes=10/data_0.parquet

Columnas y tipos:
flujo: Int32
nivel_taric: Int32
cod_taric: Float64
pais: Int32
euros: Float64

Primeras filas:
shape: (10, 5)
┌───────┬─────────────┬─────────────┬──────┬───────────┐
│ flujo ┆ nivel_taric ┆ cod_taric   ┆ pais ┆ euros     │
│ ---   ┆ ---         ┆ ---         ┆ ---  ┆ ---       │
│ i32   ┆ i32         ┆ f64         ┆ i32  ┆ f64       │
╞═══════╪═════════════╪═════════════╪══════╪═══════════╡
│ 0     ┆ 4           ┆ 3.919108e7  ┆ 664  ┆ 194753.54 │
│ 1     ┆ 4           ┆ 8.04209e6   ┆ 600  ┆ 2844.0    │
│ 1     ┆ 4           ┆ 2.2030001e7 ┆ 54   ┆ 15379.84  │
│ 1     ┆ 4           ┆ 5.702929e7  ┆ 61   ┆ 210.4     │
│ 1     ┆ 5           ┆ 8.4818e9    ┆ 93   ┆ 1823.41   │
│ 1     ┆ 3           ┆ 321310.0    ┆ 220  ┆ 2455.96   │
│ 1     ┆ 5           ┆ 8.4439e9    ┆ 5    ┆ 475138.94 │
│ 0     ┆ 4           ┆ 8.701949e7  ┆ 6    ┆ 130134.0 

flujo,nivel_taric,cod_taric,pais,euros
i32,i32,f64,i32,f64
0,4,3.919108e7,664,194753.54
1,4,8.04209e6,600,2844.0
1,4,2.2030001e7,54,15379.84
1,4,5.702929e7,61,210.4
1,5,8.4818e9,93,1823.41
…,…,…,…,…
1,4,7.094e6,64,106723.9
0,4,3.405909e7,4,131137.64
0,5,8.5098e9,17,29735.45
